<a href="https://colab.research.google.com/github/vituhaa/Healthy-Posture/blob/project/keras_openpose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проведение иференса модели OpenPose с соответствующими выводами, а также проведение процесса дообучения (fine-tuning) модели.

Проверка ресурсов nvidia для дальнейшего использования в работе с OpenPose

In [ ]:
! nvcc --version
! nvidia-smi

Скачиваю необходимые зависимости для OpenPose

In [ ]:
# ! sudo apt-get --assume-yes update
! sudo apt-get --assume-yes install build-essential
! sudo apt-get --assume-yes install libopencv-dev
! sudo apt-get --assume-yes install libatlas-base-dev libprotobuf-dev libleveldb-dev libsnappy-dev libhdf5-serial-dev protobuf-compiler
! sudo apt-get --assume-yes install --no-install-recommends libboost-all-dev
! sudo apt-get --assume-yes install libgflags-dev libgoogle-glog-dev liblmdb-dev
! sudo apt-get --assume-yes install python3-setuptools python3-dev build-essential
! sudo apt-get --assume-yes install python3-pip
! sudo -H pip3 install --upgrade numpy protobuf opencv-python
! sudo apt-get --assume-yes install opencl-headers ocl-icd-opencl-dev
! sudo apt-get --assume-yes install libviennacl-dev


Перехожу в рабочую папку и клонирую репозиторий OpenPose

In [ ]:
%%bash
cd /content
git clone https://github.com/CMU-Perceptual-Computing-Lab/openpose
cd openpose && git submodule update --init --recursive --remote

In [ ]:
! cmake --version

Скачиваю веса модели COCO (https://huggingface.co/camenduru/openpose/tree/main/models/pose/coco), так как их нет в папке с моделью

In [ ]:
%%bash
ls -lah /content/openpose/models/pose/coco/

Провожу операцию cmake внутри папки build, указывая флаги того, что нужно использовать

In [ ]:
%%bash
cd /content/openpose
mkdir build
cd build
cmake .. \
  -DBUILD_PYTHON=OFF \
  -DBUILD_EXAMPLES=ON \
  -DGPU_MODE=CUDA \
  -DUSE_CUDNN=OFF

make -j"$(nproc)"

Создаю директорию output, чтобы провести инференс модели на фото, которые находятся внутри папки examples/media. В output складываются фотографии с размеченными скелетами и .json файлы с аннотяциями. Инференс выполняется с моделью COCO.

In [ ]:
%%bash
set -e
cd /content/openpose/
# mkdir output

./build/examples/openpose/openpose.bin \
  --image_dir examples/media/ \
  --write_images output/ \
  --write_json output/ \
  --display 0 \
  --render_pose 1 \
  --model_pose COCO

ls -lah /content/openpose/output

Вывод фото в ноутбук

In [ ]:
from matplotlib import pyplot as plt
from PIL import Image

res = Image.open('/content/openpose/output/COCO_val2014_000000000589_rendered.png')
plt.figure(figsize=(8, 8))
plt.imshow(res)
plt.show()

Инференс модели OpenPose прошёл успешно на базовых фотографиях, заложенных в репозитории. Проведём подобный эксперимент на 80 фотографиях сидячих людей и оценим результаты визуально.

In [ ]:
%%bash
cd /content/openpose/
# mkdir output

./build/examples/openpose/openpose.bin \
  --image_dir examples/media/custom_sitting_dataset \
  --write_images output/sitting_output_images/ \
  --write_json output/sitting_output_jsons/ \
  --display 0 \
  --render_pose 1 \
  --model_pose COCO

ls -lah /content/openpose/output

In [ ]:
%%bash
ls -lah /content/openpose/output/sitting_output_images/
ls -lah /content/openpose/output/sitting_output_jsons/

In [ ]:
from pathlib import Path

rendered_photos = Path("/content/openpose/output/sitting_output_images")
for img in rendered_photos.rglob("*frame*"):
  if img in rendered_photos.rglob("*png*"):
    res = Image.open(img)
    plt.figure(figsize=(8, 8))
    plt.imshow(res)
    plt.show()


Визуально модель хорошо справляется с определением ключевых точек на теле сидячего человека. Тем не менее, заметно, что некоторые точки не определяются верно или не определяются вовсе. Сделаем вывод на основе метрик качества.

## Покажем качество модели на фото сидячих людей.

Попробуем обработать файлы с помощью инструмента COCOeval из библиотеки pycocotools. Для этого нужно скачать все .json файлы, которые создала модель для 80 фотографий, объединить в один и сравнить с одним файлом .json в COCO-формате, в котором находится рукотворная разметка фотографий, созданная в приложении Label Studio.

Фотографии были экспортированы в гугл колаб в том же порядке, в котором были обработаны в Label Studio.
Основная проблема инструмента COCOeval заключается в том, что на вход принимаются .json файлы с одинаковым количеством ключевых точек, в то время как OpenPose с флагом write_coco_json (COCO-формат) создаёт только 17 точек без шеи. Поэтому использовался флаг write_json, создающий 18 точек.

Пример по json файлу одной фотографии, который сгенерировал OpenPose.

In [ ]:
arr = [507.973,331.592,0.754169,590.213,599.955,0.321802,351.16,619.557,
       0.272292,0,0,0,0,0,0,725.361,490.289,0.0630654,0,0,0,0,0,0,0,0,0,
       0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,462.945,321.814,0.864411,541.177,292.46,
       0.788628,431.463,384.494,0.435692,615.68,304.188,0.680305]
print(len(arr) / 3)

Соединим json файлы в один:

In [ ]:
import os

jsons = os.listdir('/content/openpose/output/sitting_output_jsons')
k = 0
for path in jsons:
  if os.path.isfile(os.path.join('/content/openpose/output/sitting_output_jsons', path)):
    k += 1
print("JSON-файлов получилось: ", k)

rendered_images = os.listdir('/content/openpose/output/sitting_output_images')
p = 0
for path in rendered_images:
  if os.path.isfile(os.path.join('/content/openpose/output/sitting_output_images', path)):
    p += 1
print("Обработанных фото получилось: ", p)

In [ ]:
# прохожусь по всем json файлам и мёрджу
import json
import os

files = os.listdir('/content/openpose/output/sitting_output_jsons')
out = sorted(files)
arr_jsons = []
merged_jsons = []
# print(out)
data = []
for x in out:
  if 'json' in x:
    arr_jsons.append(x)
for x in arr_jsons:
  with open('/content/openpose/output/sitting_output_jsons/' + x, 'r', encoding='utf-8') as in_file:
    info = json.load(in_file)
    data.append(info)
with open('openpose_result.json', 'w', encoding='utf-8') as out_file:
    json.dump(data, out_file, indent=4)

В процессе работы возникла проблема с количеством "pose_keypoints_2d" в выходном файле JSON, в котором объеденены другие файлы JSON 80-ти фотографий: их получилось больше, чем предполагалось.

In [ ]:
find = ["pose_keypoints_2d"]
count = 0
with open("openpose_result.json", "r", encoding="utf-8") as in_file:
  info = json.load(in_file)

stack = [info]
while stack:
  x = stack.pop()
  if isinstance(x, dict):
    for k in find:
      if k in x:
        count += 1
    stack.extend(x.values())
  elif isinstance(x, list):
    stack.extend(x)

print(count)

87


In [ ]:
find = ["keypoints"]
count = 0
with open("result.json", "r", encoding="utf-8") as in_file:
  info = json.load(in_file)

stack = [info]
while stack:
  x = stack.pop()
  if isinstance(x, dict):
    for k in find:
      if k in x:
        count += 1
    stack.extend(x.values())
  elif isinstance(x, list):
    stack.extend(x)

print(count)

81


В файле разметки из Label Studio лишний "keypoints" появился из-за того, что в начале файла есть объявление всех ключевых точек в проекте в целом, поэтому это можно не учитывать. Тем не менее, в JSON файле 7 лишних объявлений ключевых точек.

Файл был осмотрен вручную в сопоставлении с полученными фотографиями. Проблема обнаружена: OpenPose на некоторых фотографиях ложно определяет двух людей. В таком случае, для чистоты эксперимента, лишние столбцы "pose_keypoints_2d" были удалены вручную.

Также при удалении лишних ключевых точек, опознанных как второй человек на фотографии, было замечено, что некоторые фотографии OpenPose вовсе не разметил, в таком случае принято решение оформить столбец с ключевыми точками в виде нулей.

In [ ]:
find = ["pose_keypoints_2d"]
count = 0
with open("final.json", "r", encoding="utf-8") as in_file:
  info = json.load(in_file)

stack = [info]
while stack:
  x = stack.pop()
  if isinstance(x, dict):
    for k in find:
      if k in x:
        count += 1
    stack.extend(x.values())
  elif isinstance(x, list):
    stack.extend(x)

print(count)

80


Как видно, теперь всё в порядке.

В статье https://www.pythontutorials.net/blog/coco-api-evaluation-for-subset-of-classes/ сказано, что формат predictions должен быть следующим:
1) image_id
2) category_id
3) bbox/keypoints/area
4) score

In [ ]:
# отредактируем остальные части файла с помощью кода
import json

res_file = [] # сюда будут записываться результаты изменённого файла
with open("final.json", "r", encoding="utf-8") as in_file: # открываю файл, читаю
  info = json.load(in_file)
inner_count = 0
for x in info:
  # print(x)
  for k in x.get("people"):
    # print(k) # словарь из person_id, image_id, pose_keypoints_2d и тд
    inner_keypoints = k.get("pose_keypoints_2d") # список ключевых точек
    # print(inner_keypoints)
    inner_image_id = k.get("image_id")
    category_id = 1 # всегда равна 1, так как в разметке из label studio ключевые точки лежат под категорией 1
    lst_confidence = inner_keypoints[2:len(inner_keypoints):3]
    # print(lst_confidence)
    for i in range (2, len(inner_keypoints), 3):
      if (inner_keypoints[i] != 0):
        inner_keypoints[i] = 2 # visibility
    sum_confidence = 0
    len_confidence = 0
    for i in range(len(lst_confidence)):
      if lst_confidence[i] != 0:
        sum_confidence += lst_confidence[i]
        len_confidence += 1
    if len_confidence == 0:
      score = 0
    else:
      score = sum_confidence / len_confidence # среднее по ненулевым confidence из списка ключевых точек
    # print(score)
    res_file.append({"image_id": inner_image_id, "category_id": category_id, "keypoints": inner_keypoints, "score": score})

# result_file_keys = ["annotations"]
# result_file = dict(zip(result_file_keys, [res_file]))
# print(res_file)
with open('openpose_result.json', 'w', encoding='utf-8') as out_file:
  json.dump(res_file, out_file, indent=4)


In [ ]:
with open("openpose_result.json", "r", encoding="utf-8") as in_file: # открываю файл, читаю
  info = json.load(in_file)
arr = []
for x in info:
  arr.append(x.get("keypoints"))
print("Количество фотографий:", len(arr))
print("Количество ключевых точек для каждой фотографии:")
for x in arr:
  print(len(x) / 3)

In [ ]:
# проверка
find = ["keypoints"]
count = 0
with open("openpose_result.json", "r", encoding="utf-8") as in_file:
  info = json.load(in_file)

stack = [info]
while stack:
  x = stack.pop()
  if isinstance(x, dict):
    for k in find:
      if k in x:
        count += 1
    stack.extend(x.values())
  elif isinstance(x, list):
    stack.extend(x)

print(count)

Также отредактируем файл с показательной разметкой:

In [ ]:
# id
# image_id
# category_id
# segmentation
# ignore
# iscrowd
# area
# keypoints
# num_keypoints
# bbox (который после keypoints)
ideal_annotations = []
with open("result.json", "r", encoding="utf-8") as in_file:
  info = json.load(in_file)
arr_keypoints = []
category_id = 1
inner_segmentation = []
ignore = 0
iscrowd = 0
inner_id = 0
arr_num_keypoints = []
for x in info.get("annotations"):
  for c in x:
    if c == "num_keypoints":
      i_num_keypoints = x.get("num_keypoints")
      arr_num_keypoints.append(i_num_keypoints)
i = 0
arr_bbox = []
for x in info.get("annotations"):
  for c in x:
    if c == "bbox":
      j_arr_bbox = x.get("bbox")
      arr_bbox.append(j_arr_bbox)
j = 0
lst_bbox = arr_bbox[1:len(arr_bbox):2]
for x in info.get("annotations"):
  # print(inner_area)
  for c in x:
    if c == "area":
      inner_area = x.get("area")
    if c == "keypoints":
      inner_keypoints = x.get("keypoints")
      ideal_annotations.append({"id": inner_id,
                                "image_id": inner_id,
                                "category_id": category_id,
                                "segmentation": inner_segmentation,
                                "ignore": ignore,
                                "iscrowd": iscrowd,
                                "area": inner_area,
                                "keypoints": inner_keypoints,
                                "num_keypoints": arr_num_keypoints[i],
                                "bbox": lst_bbox[j]})
      inner_id += 1
      i += 1
      j += 1

result_file_keys = ["images", "categories", "annotations", "info"]
result_file = dict(zip(result_file_keys,[info.get("images"), info.get("categories"),
                                         ideal_annotations, info.get("info")]))
# result_file.update(info.get("images"))
# print(result_file)

with open('ideal_annotations.json', 'w', encoding='utf-8') as out_file:
  json.dump(result_file, out_file, indent=4)
# print(ideal_annotations)
# print(len(arr_bbox[1:len(arr_bbox):2]))
# print(arr_num_keypoints)

# print(len(ideal_annotations))
# print(arr_keypoints)
# print(arr_image_id)
# print(arr_bbox_all)

In [ ]:
ls

In [ ]:
with open("ideal_annotations.json", "r", encoding="utf-8") as in_file: # открываю файл, читаю
  info = json.load(in_file)
arr = []
for x in info.get("annotations"):
  arr.append(x.get("keypoints"))
print("Количество фотографий:", len(arr))
print("Количество ключевых точек для каждой фотографии:")
for x in arr:
  print(len(x) / 3)

То есть, размерности совпадают, но есть ошибка ValueError: operands could not be broadcast together with shapes (18,) (17,).

In [ ]:
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO
import numpy as np

ground_truth = COCO('/content/ideal_annotations.json')
predictions = ground_truth.loadRes('/content/openpose_result.json')

print(len(ground_truth.dataset['categories']))
print(len(predictions.dataset['categories']))

print(len(ground_truth.dataset['images']))
print(len(predictions.dataset['images']))

print(len(ground_truth.dataset['annotations']))
print(len(predictions.dataset['annotations']))

print((ground_truth.dataset['annotations']))
print((predictions.dataset['annotations']))

sigmas_original = np.array([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072,
                   0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089])
sigmas_edited = np.concatenate([sigmas_original, [0.05]])
print(sigmas_edited)
eval = COCOeval(ground_truth, predictions, "keypoints")
# print(eval.params.kpt_oks_sigmas) # [0.026 0.025 0.025 0.035 0.035 0.079 0.079 0.072 0.072 0.062 0.062 0.107 0.107 0.087 0.087 0.089 0.089]
eval.params.kpt_oks_sigmas = sigmas_edited
eval.evaluate()
eval.accumulate()
eval.summarize()


In [ ]:
# https://github.com/AhmetEkiz/using_pycocotools/blob/main/pycocotools_detailed_getting_started.ipynb - здесь написано как вывести точки и bbox

Вывод:

## Попытаемся дообучить модель OpenPose на датасете сидячих людей, чтобы модель лучше предсказывала нахождение ключевых точек на теле людей и сделаем вывод.

Вывод:

## Создание нейронной сети:

In [ ]:
# показываю датасет:
class_names = ['good', 'bad']

In [ ]:
import keras
from keras import layers # 19 layers

input = keras.Input(shape=(64, 64, 3)) # посмотреть как писать размерность
x = keras.layers.Dense( , activation="relu")(input)
pooling = keras.layer.Pooling2D()()

output = keras.layers.Dense( , activation="softmax")(x) # вывод
keras_model = keras.Model(inputs=input, outputs=output)